# Chinese Understanding Benchmark v9

Compare:

1. `mlx-community/gemma-4-e4b-it-4bit` — MLX quantized Gemma E4B-it, evaluated first
2. `Qwen/Qwen3-4B-Instruct-2507`
3. `google/gemma-4-E2B-it`

The notebook evaluates CLUE tasks with strict label parsing and avoids reloading the same model for every task.


## 1. Install dependencies

For the MLX Gemma E4B 4-bit checkpoint, use the pinned MLX versions below if you hit `Received xxx parameters not in model`.

If `torchaudio` complains about Torch version conflicts, uninstall it; this project does not use audio.


In [3]:
# Recommended dependency setup
# Run this once, then restart the Jupyter kernel.

!pip uninstall -y torchaudio torchvision mlx-vlm || true
!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece
!pip install -U mlx==0.31.1 mlx-lm==0.31.2 mlx-metal==0.31.1


## 2. Configuration

In [4]:
# Evaluate Gemma E4B 4-bit first.
# This lets you fail fast if the quantized MLX model has compatibility problems.

MODELS = {
    "gemma_e4b_it_4bit": {
        "model_id": "mlx-community/gemma-4-e4b-it-4bit",
        "backend": "mlx",
    },
    "qwen3_4b_instruct_2507": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507",
        "backend": "transformers",
    },
    "gemma_e2b_it": {
        "model_id": "google/gemma-4-E2B-it",
        "backend": "transformers",
    },
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_SAMPLES = 200

# Print a few raw model outputs per model/task for debugging.
DEBUG_N = 3


## 3. Imports and cleanup helpers

In [5]:
import gc
import os
import re
import time
import warnings

import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


def cleanup_memory():
    """Release Python and PyTorch/MPS memory between models."""
    gc.collect()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def hard_cleanup_model(model=None, tokenizer=None):
    """Delete model/tokenizer references and clean memory."""
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    cleanup_memory()


## 4. Label mappings and prompts

Important: AFQMC and CMNLI label names from `dataset.features` are not always useful for scoring.  
We use explicit task mappings for AFQMC and CMNLI, and derive TNEWS names from the dataset.


In [6]:
# Explicit dataset label mappings for CLUE tasks.
# These match the Hugging Face CLUE label IDs used in the notebook results you showed.

TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {
            "0": "不同",
            "1": "相同",
        },
        "label_name_to_id": {
            "不同": "0",
            "相同": "1",
        },
        "aliases": {
            "不同": ["不同", "不相同", "不一致", "不等价", "语义不同", "否", "no", "false", "0"],
            "相同": ["相同", "一致", "等价", "同义", "语义相同", "是", "yes", "true", "1"],
        },
    },
    "cmnli": {
        # Based on your observed dataset:
        # 0 = neutral, 1 = entailment, 2 = contradiction
        "label_id_to_name": {
            "0": "中立",
            "1": "蕴含",
            "2": "矛盾",
        },
        "label_name_to_id": {
            "中立": "0",
            "蕴含": "1",
            "矛盾": "2",
        },
        "aliases": {
            "中立": ["中立", "无法判断", "无关", "neutral", "0"],
            "蕴含": ["蕴含", "包含", "推出", "entailment", "entails", "1"],
            "矛盾": ["矛盾", "冲突", "contradiction", "contradict", "2"],
        },
    },
}


def make_tnews_spec(dataset):
    """Derive TNEWS label mapping from dataset.features['label'].names."""
    names = dataset.features["label"].names
    return {
        "label_id_to_name": {str(i): name for i, name in enumerate(names)},
        "label_name_to_id": {name: str(i) for i, name in enumerate(names)},
        "aliases": {name: [name] for name in names},
    }


def get_task_spec(task, dataset=None):
    if task == "tnews":
        if dataset is None:
            raise ValueError("TNEWS needs the loaded dataset to infer label names.")
        return make_tnews_spec(dataset)
    return TASK_SPECS[task]


def label_id_to_name(task, label_id, spec):
    return spec["label_id_to_name"].get(str(label_id), str(label_id))


def normalize_text(text):
    text = str(text).strip().lower()
    text = text.replace("答案：", "").replace("答案:", "")
    text = text.replace("标签：", "").replace("标签:", "")
    text = text.replace("\n", " ")
    text = re.sub(r"[。，“”，、；;:：\[\]\(\)（）\"'`]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pred_id(task, raw_output, spec):
    """Map model raw output to a dataset label ID, or __invalid__."""
    text = normalize_text(raw_output)

    # Exact normalized alias match first.
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            if normalize_text(alias) == text:
                return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    # Substring match.
    # Sort longer aliases first to avoid accidental short matches.
    alias_items = []
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            alias_items.append((canonical_name, normalize_text(alias)))
    alias_items = sorted(alias_items, key=lambda x: len(x[1]), reverse=True)

    for canonical_name, alias_norm in alias_items:
        if alias_norm and alias_norm in text:
            return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    return "__invalid__"


def pred_id_to_name(pred_id, spec):
    if pred_id == "__invalid__":
        return "__invalid__"
    return spec["label_id_to_name"].get(str(pred_id), "__invalid__")


In [7]:
def build_prompt(task, ex, spec):
    labels = "、".join(spec["label_name_to_id"].keys())

    if task == "afqmc":
        return f"""你是中文二分类器。只输出“相同”或“不同”其中一个词，不要解释。

句子1：{ex["sentence1"]}
句子2：{ex["sentence2"]}

语义是否相同？答案："""

    if task == "cmnli":
        return f"""你是中文自然语言推理分类器。只输出“蕴含”、“中立”或“矛盾”其中一个词，不要解释。

前提：{ex["sentence1"]}
假设：{ex["sentence2"]}

关系是？答案："""

    if task == "tnews":
        return f"""你是中文新闻标题分类器。只能从以下类别中选择一个输出，不要解释。

可选类别：{labels}

标题：{ex["sentence"]}

类别："""

    raise ValueError(f"Unsupported task: {task}")


## 5. Backends: Transformers and MLX

In [8]:
def load_transformers_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    return tokenizer, model


@torch.no_grad()
def generate_transformers(tokenizer, model, prompt, max_new_tokens=8):
    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the model's main device.
    # With device_map="auto", this usually works for generation.
    try:
        inputs = inputs.to(model.device)
    except Exception:
        pass

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()


In [9]:
# MLX imports are kept separate so the Transformers-only part can still run
# even if MLX installation has a problem.

def load_mlx_model(model_id):
    from mlx_lm import load as mlx_load
    model, tokenizer = mlx_load(model_id)
    return tokenizer, model


def generate_mlx(tokenizer, model, prompt, max_new_tokens=8):
    from mlx_lm import generate as mlx_generate
    return mlx_generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_new_tokens,
        verbose=False,
    ).strip()


## 6. Evaluation loop

Each model is loaded once, evaluated on all tasks, then explicitly cleared before the next model.


In [11]:
def evaluate_loaded_model(model_key, model_id, backend, tokenizer, model, tasks=TASKS):
    all_rows = []
    summaries = []

    for task in tasks:
        dataset = load_dataset("clue", task, split=SPLIT)
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
        spec = get_task_spec(task, dataset)

        y_true = []
        y_pred = []

        start = time.time()

        for idx, ex in enumerate(tqdm(dataset, desc=f"{model_key}/{task}")):
            prompt = build_prompt(task, ex, spec)

            if backend == "transformers":
                raw = generate_transformers(tokenizer, model, prompt, max_new_tokens=8)
            elif backend == "mlx":
                raw = generate_mlx(tokenizer, model, prompt, max_new_tokens=8)
            else:
                raise ValueError(f"Unsupported backend: {backend}")

            gold_id = str(ex["label"])
            gold_name = label_id_to_name(task, gold_id, spec)

            pred_id = extract_pred_id(task, raw, spec)
            pred_name = pred_id_to_name(pred_id, spec)

            y_true.append(gold_id)
            y_pred.append(pred_id)

            row = {
                "model_key": model_key,
                "model_id": model_id,
                "backend": backend,
                "task": task,
                "split": SPLIT,
                "idx": idx,
                "gold_id": gold_id,
                "gold_name": gold_name,
                "raw": repr(raw),
                "pred_name": pred_name,
                "pred_id": pred_id,
            }
            all_rows.append(row)

            if idx < DEBUG_N:
                print(row)

        seconds = time.time() - start
        invalid_count = sum(p == "__invalid__" for p in y_pred)

        # Treat invalid predictions as wrong by mapping to a nonexistent class.
        y_pred_for_score = [p if p != "__invalid__" else "-1" for p in y_pred]

        summary = {
            "model_key": model_key,
            "model_id": model_id,
            "backend": backend,
            "task": task,
            "split": SPLIT,
            "samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred_for_score),
            "macro_f1": f1_score(y_true, y_pred_for_score, average="macro", zero_division=0),
            "invalid_rate": invalid_count / len(y_pred),
            "invalid_count": invalid_count,
            "seconds": seconds,
            "samples_per_second": len(y_true) / seconds if seconds > 0 else None,
        }

        print(summary)
        summaries.append(summary)

    return summaries, all_rows

In [12]:
def evaluate_one_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    backend = model_cfg["backend"]

    cleanup_memory()
    print(f"\n=== Loading {model_key}: {model_id} [{backend}] ===")

    tokenizer = None
    model = None

    try:
        if backend == "transformers":
            tokenizer, model = load_transformers_model(model_id)
        elif backend == "mlx":
            tokenizer, model = load_mlx_model(model_id)
        else:
            raise ValueError(f"Unsupported backend: {backend}")

        summaries, rows = evaluate_loaded_model(
            model_key=model_key,
            model_id=model_id,
            backend=backend,
            tokenizer=tokenizer,
            model=model,
        )

    finally:
        print(f"Clearing model from memory: {model_key}")
        hard_cleanup_model(model, tokenizer)

    return summaries, rows


## 7. Run benchmark

Because Gemma E4B 4-bit is first, you can stop early if it fails to load.


In [13]:
all_summaries = []
all_rows = []

for model_key, model_cfg in MODELS.items():
    summaries, rows = evaluate_one_model(model_key, model_cfg)
    all_summaries.extend(summaries)
    all_rows.extend(rows)

summary_df = pd.DataFrame(all_summaries)
detail_df = pd.DataFrame(all_rows)

summary_df



=== Loading gemma_e4b_it_4bit: mlx-community/gemma-4-e4b-it-4bit [mlx] ===


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

gemma_e4b_it_4bit/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '不同', 'raw': "'双十一花呗提额在哪'", 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 1, 'gold_id': '0', 'gold_name': '不同', 'raw': "'花呗付款\\n\\n句子1：花'", 'pred_name': '相同', 'pred_id': '1'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 2, 'gold_id': '1', 'gold_name': '相同', 'raw': "'我到支付宝实体店消费用花'", 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'samples': 200, 'accuracy': 0.185, 'macro_f1': 0.14883913764510778, 'invalid_ra

gemma_e4b_it_4bit/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

...

gemma_e2b_it/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 0, 'gold_id': '2', 'gold_name': '102', 'raw': "'10000000'", 'pred_name': '100', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 1, 'gold_id': '9', 'gold_name': '110', 'raw': "'100\\n\\n标题：以色列大规模'", 'pred_name': '100', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 2, 'gold_id': '4', 'gold_name': '104', 'raw': "'100'", 'pred_name': '100', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'samples': 200, 'accuracy': 0.01, 'macro_f1': 0.01851503759398496, 'invalid_rate': 0.065, 'invalid_count': 13, 'seconds': 114.70266485214233, 'samples

gemma_e2b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '中立', 'raw': "'每个人都很喜欢最新的福利\\n\\n答案：'", 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 1, 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'嗯，我不知道，我不知道，'", 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 2, 'gold_id': '2', 'gold_name': '矛盾', 'raw': "'我最喜欢的餐馆总是离我家'", 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'samples': 200, 'accuracy': 0.025, 'macro_f1': 0.030277306168647426, 'invalid_rate'

,model_key,model_id,backend,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second
0,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,afqmc,validation,200,0.185,0.148839,0.490,98,105.591305,1.894095
1,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,tnews,validation,200,0.005,0.005000,0.740,148,120.345907,1.661876
2,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,cmnli,validation,200,0.035,0.047343,0.900,180,111.336252,1.796360
3,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,afqmc,validation,200,0.670,0.658950,0.000,0,178.860442,1.118190
4,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,tnews,validation,200,0.055,0.028041,0.000,0,198.616628,1.006965
5,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,cmnli,validation,200,0.740,0.736331,0.000,0,143.646617,1.392306
6,gemma_e2b_it,google/gemma-4-E2B-it,transformers,afqmc,validation,200,0.045,0.066241,0.890,178,108.208538,1.848283
7,gemma_e2b_it,google/gemma-4-E2B-it,transformers,tnews,validation,200,0.010,0.018515,0.065,13,114.702665,1.743639
8,gemma_e2b_it,google/gemma-4-E2B-it,transformers,cmnli,validation,200,0.025,0.030277,0.955,191,111.059383,1.800838


In [14]:
summary_df

,model_key,model_id,backend,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second
0,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,afqmc,validation,200,0.185,0.148839,0.490,98,105.591305,1.894095
1,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,tnews,validation,200,0.005,0.005000,0.740,148,120.345907,1.661876
2,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,cmnli,validation,200,0.035,0.047343,0.900,180,111.336252,1.796360
3,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,afqmc,validation,200,0.670,0.658950,0.000,0,178.860442,1.118190
4,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,tnews,validation,200,0.055,0.028041,0.000,0,198.616628,1.006965
5,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,cmnli,validation,200,0.740,0.736331,0.000,0,143.646617,1.392306
6,gemma_e2b_it,google/gemma-4-E2B-it,transformers,afqmc,validation,200,0.045,0.066241,0.890,178,108.208538,1.848283
7,gemma_e2b_it,google/gemma-4-E2B-it,transformers,tnews,validation,200,0.010,0.018515,0.065,13,114.702665,1.743639
8,gemma_e2b_it,google/gemma-4-E2B-it,transformers,cmnli,validation,200,0.025,0.030277,0.955,191,111.059383,1.800838


In [ ]:
detail_df.head()


In [ ]:
invalid_df = detail_df[detail_df["pred_id"] == "__invalid__"].copy()
print("Invalid count:", len(invalid_df))
invalid_df.head(30)


In [ ]:
# Save results
os.makedirs("results", exist_ok=True)
summary_df.to_csv("results/chinese_understanding_summary_v9.csv", index=False)
detail_df.to_csv("results/chinese_understanding_details_v9.csv", index=False)

print("Saved:")
print("results/chinese_understanding_summary_v9.csv")
print("results/chinese_understanding_details_v9.csv")


## 8. Optional: LoRA fine-tuning scaffold

This section is for Transformers models only. MLX fine-tuning is not included here.

For your 32GB Mac, do not full fine-tune these models. Use LoRA only, keep samples small, and restart the kernel before training.


In [ ]:
# Optional: LoRA fine-tuning scaffold for Transformers models only.
# This cell is intentionally not run by default.

RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    from peft import LoraConfig
    from trl import SFTTrainer
    from transformers import TrainingArguments

    def format_sft_example(task, ex, spec):
        prompt = build_prompt(task, ex, spec)
        answer = label_id_to_name(task, ex["label"], spec)
        return prompt + answer

    def train_lora(model_id, task="afqmc", output_dir="adapters/lora_model", max_train_samples=1000):
        dataset = load_dataset("clue", task, split="train")
        dataset = dataset.select(range(min(max_train_samples, len(dataset))))
        spec = get_task_spec(task, dataset)

        dataset = dataset.map(lambda ex: {"text": format_sft_example(task, ex, spec)})

        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        )

        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            num_train_epochs=1,
            logging_steps=20,
            save_steps=500,
            fp16=True,
            report_to="none",
        )

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            peft_config=lora_config,
            args=training_args,
            max_seq_length=512,
        )

        trainer.train()
        trainer.save_model(output_dir)

        hard_cleanup_model(model, tokenizer)

    # Example:
    # train_lora("Qwen/Qwen3-4B-Instruct-2507", task="afqmc", output_dir="adapters/qwen3_4b_afqmc")
